# QSAR Modelling of Human Dopamine Receptor D2 (DRD2) Ligand Potency

**An end-to-end, reproducible ligand-based potency model — from a live ChEMBL query to a calibrated, interpretable predictor with a stated applicability domain.**

Babak Mamnoon, Ph.D. · Computational Drug Discovery / Cheminformatics

---

### What this notebook is

This is the narrative layer over a tested Python package (`src/drd2_qsar/`). Every
computation you see here runs inside that package, so the same code path produces
the figures in this notebook, the numbers in the README, and the predictions from
the command-line inference script. The notebook explains; it does not reimplement.

    pip install -e .
    python -m drd2_qsar.run --config configs/drd2.yaml     # the whole thing, one command

### What it is not

It is not a demonstration that gradient boosting can fit a fingerprint. That part is
four lines and it is the least interesting part of a QSAR project. The substance is in
the parts that decide whether the number at the end means anything:

| Stage | The question it answers |
|---|---|
| Curation | Is the data physically possible, and does each row mean what the column says? |
| Scaffold splitting | Does the score measure generalisation, or memorisation of a congeneric series? |
| Baseline ladder | Does the model beat a similarity lookup table? |
| Confidence intervals | Is the difference between two models larger than the noise? |
| Y-scrambling | Is the model fitting signal, or fitting the split? |
| Applicability domain | Where is this model allowed to be used? |
| Conformal prediction | How wrong is a given prediction likely to be? |

### Contents

0. [Setup and provenance](#0)
1. [Data ingestion — a live ChEMBL query, not a CSV](#1)
2. [Curation — where the real defects are](#2)
3. [Dataset characterisation](#3)
4. [Partitioning — why scaffold, not random](#4)
5. [Featurization and the baseline ladder](#5)
6. [Molecular-representation ablation](#6)
7. [Validation: nested CV, y-scrambling, split stability](#7)
8. [Applicability domain and conformal prediction](#8)
9. [Interpretation: SHAP and activity cliffs](#9)
10. [Results summary, limitations, and what I would do next](#10)


<a id='0'></a>
## 0. Setup and provenance

The environment, the config hash and the ChEMBL release are all recorded, because a QSAR number without them cannot be reproduced or compared.

In [ ]:
import importlib
import importlib.util
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

REPO = "QSAR-Modelling-of-Human-Dopamine-Receptor-D2-DRD2-Ligand-Potency"
REPO_URL = f"https://github.com/Babakmamnoon/{REPO}.git"
CSV_URL = f"https://raw.githubusercontent.com/Babakmamnoon/{REPO}/main/DRD2.csv"
ZIP_NAME = "drd2-qsar-project.zip"

IN_COLAB = "google.colab" in sys.modules


def _is_project(p: Path) -> bool:
    """A directory is the project root iff it carries the package and its config."""
    return (p / "pyproject.toml").is_file() and (p / "src" / "drd2_qsar").is_dir()


def _install(root: Path) -> Path:
    print(f"installing drd2_qsar from {root} ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True
    )
    src = str((root / "src").resolve())
    if src not in sys.path:
        sys.path.insert(0, src)
    importlib.invalidate_caches()
    return root


def _unpack(zip_path: Path, into: Path) -> Path | None:
    """Extract the project zip and return the extracted project root."""
    print(f"unpacking {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(into)
    for cand in [into, *sorted(p for p in into.iterdir() if p.is_dir())]:
        if _is_project(cand):
            return cand
    return None


def _search_dirs() -> list[Path]:
    """Places the project or its zip plausibly sits, in priority order."""
    here = Path.cwd()
    cands = [here, *here.parents, Path("/content"), Path("/content/drive/MyDrive"),
             Path.home(), Path.home() / "Downloads", Path.home() / "Desktop"]
    seen, out = set(), []
    for c in cands:
        try:
            r = c.resolve()
        except OSError:
            continue
        if r not in seen and r.is_dir():
            seen.add(r)
            out.append(r)
    return out


def bootstrap() -> Path:
    """Make `drd2_qsar` importable and return the project root.

    The notebook is only the narrative layer — every computation lives in the
    package — so nothing below cell 0 can run until the package is on the path.
    This resolves that from whatever the environment happens to offer, in
    descending order of convenience:

      1. already importable, with the project root locatable;
      2. an unpacked project in or above the working directory, or in one of the
         usual Colab / Drive / Downloads locations;
      3. a `drd2-qsar-project.zip` in any of those locations;
      4. the GitHub repository, if the package has been pushed to it;
      5. in Colab, an interactive upload prompt for the zip.

    Only if all five fail does it raise, and then it says exactly what to do.
    """
    here = Path.cwd()

    # 1. already importable -- but only trust it if the project root is findable,
    #    otherwise `load_config('configs/drd2.yaml')` fails two cells later.
    if importlib.util.find_spec("drd2_qsar") is not None:
        for cand in (here, *here.parents):
            if (cand / "configs" / "drd2.yaml").is_file():
                return cand
        try:
            import drd2_qsar

            pkg_root = Path(drd2_qsar.__file__).resolve().parents[2]
            if (pkg_root / "configs" / "drd2.yaml").is_file():
                return pkg_root
        except Exception:
            pass
        print("drd2_qsar is importable but its configs/ was not found; re-resolving ...")

    # 2. an unpacked project nearby
    for d in _search_dirs():
        for cand in (d, d / "drd2-qsar", d / REPO):
            if _is_project(cand):
                return _install(cand)

    # 3. the project zip
    for d in _search_dirs():
        z = d / ZIP_NAME
        if z.is_file():
            root = _unpack(z, here)
            if root is not None:
                return _install(root)

    # 4. the repository
    target = here / REPO
    if not target.exists():
        print(f"cloning {REPO} ...")
        try:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(target)], check=True)
        except subprocess.CalledProcessError as exc:
            print(f"clone failed ({exc}); continuing")
    if _is_project(target):
        return _install(target)

    # 5. Colab: ask for the zip directly
    if IN_COLAB:
        print(
            "\nThe GitHub repository does not yet contain the drd2_qsar package,\n"
            f"and no local copy was found. Please upload {ZIP_NAME} now\n"
            "(the file picker below), or press Cancel to see the alternatives.\n"
        )
        try:
            from google.colab import files

            uploaded = files.upload()
            for name in uploaded:
                p = Path(name)
                if p.suffix == ".zip":
                    root = _unpack(p, here)
                    if root is not None:
                        return _install(root)
        except Exception as exc:
            print(f"upload cancelled or failed ({exc})")

    raise ModuleNotFoundError(
        f"""Could not locate the drd2_qsar package.

The notebook is only the narrative layer; every computation lives in the
package (src/drd2_qsar/, pyproject.toml, configs/drd2.yaml). The GitHub
repository currently holds only DRD2.csv, README.md and this notebook,
so cloning it is not enough.

Pick whichever is easiest:

  A. Upload the project zip. In Colab, run in a new cell:
         from google.colab import files; files.upload()
     choose {ZIP_NAME}, then re-run this cell.
     (It is also found automatically if you put it in /content, in the
     working directory, or in Google Drive at /content/drive/MyDrive.)

  B. Push the package to GitHub once, and every later run just works.
     From the unzipped project folder on your machine:
         bash push_to_github.sh
     or manually:
         git init && git add -A
         git commit -m "Add drd2_qsar package, configs and tests"
         git remote add origin {REPO_URL}
         git pull --rebase --allow-unrelated-histories origin main
         git push -u origin main

  C. Run locally instead: unzip the project, `cd drd2-qsar`, then
         pip install -e .
     and start Jupyter from that folder."""
    )


PROJECT_ROOT = bootstrap()

import os

os.chdir(PROJECT_ROOT)
print(f"project root : {PROJECT_ROOT}")

# DRD2.csv is the raw ChEMBL web-interface export. The pipeline prefers a live
# REST query and falls back to this file when www.ebi.ac.uk is unreachable, so
# stage it where chembl.load_or_fetch looks. The loader identifies it by its
# header, not its filename.
export = PROJECT_ROOT / "data" / "raw" / "DRD2_chembl_export.csv"
export.parent.mkdir(parents=True, exist_ok=True)
if not export.exists():
    local_csv = PROJECT_ROOT / "DRD2.csv"
    if local_csv.is_file():
        shutil.copy(local_csv, export)
    else:
        print("downloading DRD2.csv ...")
        urllib.request.urlretrieve(CSV_URL, export)
print(f"ChEMBL export : {export.relative_to(PROJECT_ROOT)} ({export.stat().st_size:,} bytes)")

In [ ]:
import json, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

from drd2_qsar import chembl, curate, evaluate, features, interpret, models, plots, splits
from drd2_qsar.config import load_config, set_global_seed, setup_logging, environment_manifest

warnings.filterwarnings('ignore', category=FutureWarning)
setup_logging(logging.INFO)

CFG = load_config('configs/drd2.yaml')
CFG.ensure_dirs()
set_global_seed(CFG.seed)
plots.set_style(dpi=CFG.get('plots.dpi', 300))

FIG = CFG.figure_dir
OUT = CFG.output_dir
SEED = CFG.seed

print('config fingerprint :', CFG.fingerprint())
print('global seed        :', SEED)
for k, v in environment_manifest().items():
    print(f'{k:<19}: {v}')

<a id='1'></a>
## 1. Data ingestion — a live ChEMBL query, not a CSV

The earlier version of this project began by `wget`-ing a pre-made `DRD2.csv` from
GitHub. That is convenient and it is also the single biggest reproducibility hole in
a cheminformatics repository: nobody — including the author six months later — can
say what query produced those rows, which ChEMBL release they came from, or what was
filtered out before they were written.

Here the dataset is **derived**, and the derivation is the code below. The query is
explicit about four things that quietly corrupt public-data QSAR models when left
implicit:

* `standard_relation == '='` — censored measurements (`IC50 > 10000`) are real
  information but they are *not* regression targets. Treating `>` as `=` teaches
  the model that a compound which failed to show activity has exactly the potency
  of the assay's detection limit.
* `assay_type == 'B'` — binding assays only, so functional and ADMET readouts are
  not pooled into one "potency" column.
* `standard_units == 'nM'` — with the unit actually checked rather than assumed.
* ChEMBL's own `data_validity_comment` flags are honoured. These are the curator
  warnings that catch transcription and unit errors, and they are free.

The pull is cached with a manifest recording the ChEMBL release and pull date, so
re-running the notebook does not silently pick up a different dataset when ChEMBL
ships a new version.

In [ ]:
raw, manifest = chembl.load_or_fetch(CFG['chembl'], CFG.data_dir / 'raw')

print(f"source           : {manifest.source}")
print(f"ChEMBL release   : {manifest.chembl_release}")
print(f"pulled at (UTC)  : {manifest.pulled_at_utc}")
print(f"raw activity rows: {manifest.n_rows_raw:,}")
print(f"query            : {json.dumps(manifest.query, indent=2)}")

if manifest.source == 'live':
    print('\nLive REST query against the ChEMBL web services.')
elif manifest.source == 'chembl_web_export':
    print('\nRaw activity export downloaded from the ChEMBL web interface.')
    print('This is the same content the REST query returns, so every curation')
    print('stage applies unchanged. The ChEMBL release is not recorded in the')
    print('export, which is the one piece of provenance a live pull adds.')
elif manifest.source == 'legacy_snapshot':
    print('\n' + '!' * 72)
    print('PROVENANCE WARNING: falling back to the legacy pre-curated snapshot.')
    print('It is already truncated to IC50 < 1000 nM and carries the unit errors')
    print('this pipeline is built to catch. Metrics from this run are')
    print('illustrative only; re-run with ChEMBL access or the web export.')
    print('!' * 72)

raw.head(3)

<a id='2'></a>
## 2. Curation — where the real defects are

This is the stage that changes the answer, and it is the stage most QSAR notebooks
skip. Six things go wrong with a raw ChEMBL potency table, and the audit trail below
records exactly how many rows each one costs.

**2.1 Censored measurements.** Roughly a fifth of the raw rows are reported as
`IC50 > x` — the compound did not show activity up to the highest concentration
tested. That is real information, but it is *not* a regression target: treating `>`
as `=` teaches the model that an inactive compound has exactly the potency of the
detection limit. They are dropped, and the loss is acknowledged in the limitations
rather than hidden (those rows are overwhelmingly the inactives).

**2.2 Physically impossible potencies.** The pre-curated CSV this project started
from contained pIC50 values up to **16.4** — a sub-femtomolar affinity no small
molecule achieves at a GPCR. The offenders were dopamine (14.1), norepinephrine
(13.1), serotonin (12.5) and two catechol aminotetralins: µM or mM values recorded
as nM. Because squared-error regression is dominated by its largest residuals, five
points out of 631 were bending the entire fit.

In the current raw export those rows never reach the plausibility window — ChEMBL's
own validity flags and the censored/missing filters remove them first. The window
still runs. **A guard that catches nothing today is not redundant**; it is what stops
the next export, or the next target, from reintroducing the same class of error
unnoticed.

**2.3 Salt and solvate components.** Dozens of SMILES carry `Br.` or `Cl.` prefixes.
Fingerprinting them as-is makes the hydrobromide and the free base two different
molecules with two different feature vectors and the same measured potency — pure
label noise injected by a data-cleaning omission.

**2.4 Duplicate parents under different accessions.** Deduplication has to happen on
the **InChIKey connectivity layer**, not on the ChEMBL ID, or salt forms and
stereochemically unspecified entries of the same parent survive as independent rows.

**2.5 Irreproducible replicates.** When one compound has been measured several times
across different papers and those measurements span more than a log unit, the "true"
value is not knowable from this data. Averaging them produces a confident-looking
wrong number. Dropping them is the honest move.

**2.6 Mechanistic incoherence.** Dopamine, apomorphine and quinpirole are
**agonists**. Pooling them into a model of "inhibitor potency" is a category error,
not a modelling choice — and it is exactly why those compounds also carried the
corrupt potencies above. The `action_type_policy` setting separates them, though
ChEMBL annotates mechanism for only a small minority of compounds.

In [ ]:
action_types = None
if CFG.get('chembl.fetch_action_type', True):
    if manifest.source == 'live':
        try:
            action_types = chembl.fetch_action_types(
                raw['molecule_chembl_id'].dropna().unique().tolist())
        except chembl.ChEMBLUnavailable as e:
            print('mechanism lookup unavailable:', e)
    elif 'action_type' in raw.columns and raw['action_type'].notna().any():
        # The web export already carries Action Type, so no extra lookup is needed.
        action_types = (raw[['molecule_chembl_id', 'action_type']]
                        .dropna(subset=['action_type'])
                        .drop_duplicates('molecule_chembl_id'))
        print(f"action_type from export: {len(action_types)} annotated compounds")
        print(action_types['action_type'].value_counts().to_string())

clean, audit, rejected = curate.curate(raw, CFG['curation'], action_types)

print(f'\n{len(clean):,} unique parent structures survived curation\n')
audit.to_frame()

### What was rejected, and why

Nothing disappears without a recorded reason. This is the table a reviewer asks for
and almost never gets.

In [ ]:
if len(rejected):
    print(rejected.groupby('rejection_reason').size().to_string(), '\n')
    implausible = rejected[rejected['rejection_reason'].str.contains('plausibility')]
    if len(implausible):
        print('Physically impossible potencies caught by the sanity window:')
        cols = [c for c in ['molecule_chembl_id', 'canonical_smiles', 'pchembl_value'] if c in implausible]
        display(implausible[cols].sort_values('pchembl_value', ascending=False).head(10))
else:
    print('Nothing rejected.')

In [ ]:
plots.plot_curation_waterfall(audit.to_frame(), FIG, ('png',))
display(Image(str(FIG / 'fig00_curation_waterfall.png')))

<a id='3'></a>
## 3. Dataset characterisation

Before modelling: what is actually in this set? The property panel establishes
whether the compounds are drug-like, and the potency-versus-property panel checks
whether "potency" here is really just a bulk-property effect in disguise — if
pIC50 correlated strongly with molecular weight, the model would be learning size,
not pharmacology.

In [ ]:
props = curate.compute_properties(clean['smiles'])
print(props[['MW','cLogP','TPSA','HBD','HBA','RotB','QED']].describe().round(2).to_string())
print(f"\nRule-of-5 compliant : {(props['Lipinski_violations'] == 0).mean():.1%}")
print(f"Veber compliant     : {props['Veber_pass'].mean():.1%}")
print(f"pIC50 range         : {clean['pIC50'].min():.2f} – {clean['pIC50'].max():.2f}"
      f"  (SD {clean['pIC50'].std():.2f})")

In [ ]:
plots.plot_data_overview(clean, props, FIG, ('png',))
display(Image(str(FIG / 'fig01_data_overview.png')))

In [ ]:
plots.plot_property_vs_potency(clean, props, FIG, ('png',))
display(Image(str(FIG / 'fig02_property_vs_potency.png')))

<a id='4'></a>
## 4. Partitioning — why scaffold, not random

`train_test_split(df)` — the legacy notebook's split — has three defects, and the
third is fatal.

1. No `random_state`, so the result is not reproducible.
2. No `test_size`, so it silently uses 25% while the README claimed 80/20.
3. **It is random.** ChEMBL target sets are dominated by congeneric series: dozens of
   close analogues published together in one paper. A random split scatters those
   analogues across the train/test boundary, so the test set is mostly measuring how
   well the model interpolates within series it has already memorised.

A **Bemis–Murcko scaffold split** forces whole chemical series into one partition.
The number it produces is lower, and it is the one that predicts what happens when
the model meets next quarter's chemistry.

This notebook reports **both** — and, when publication years are available, a
**temporal** split too (train on pre-2015 literature, test on what came after).
The gaps between them are treated as headline results. Hiding them is the
difference between a portfolio project and a tutorial.

The split is three-way. Validation is used for model selection and tuning; the test
set is touched exactly once, at the very end. Selecting the best model by test score
is a subtler form of leakage — the test set stops being held out the moment it
decides anything.

In [ ]:
clean = splits.add_scaffolds(clean, CFG.get('split.scaffold.include_chirality', False))

split_idx  = splits.make_split(clean, CFG['split'], seed=SEED)
random_idx = splits.random_split(clean, CFG.get('split.test_size'), CFG.get('split.val_size'), SEED)

# Hard guarantees, not hopes: these raise rather than warn.
splits.assert_no_structure_leak(clean, split_idx)

summary = splits.split_summary(clean, split_idx)
display(summary.round(3))

n_scaf = clean['scaffold'].nunique()
print(f"{n_scaf} unique scaffolds for {len(clean)} compounds "
      f"({len(clean)/n_scaf:.2f} compounds per scaffold)")
print(f"singleton scaffolds: {(clean['scaffold'].value_counts() == 1).sum()}")

In [ ]:
plots.plot_scaffold_analysis(clean, FIG, 20, ('png',))
display(Image(str(FIG / 'fig04_scaffold_analysis.png')))

In [ ]:
plots.plot_scaffold_grid(clean, FIG, 12)
display(Image(str(FIG / 'fig05_scaffold_structures.png')))

<a id='5'></a>
## 5. Featurization and the baseline ladder

A QSAR result is only interpretable relative to what a trivial method achieves on the
same split. The ladder runs from "predict the training mean" up to gradient boosting.

The rung that matters most is **1-nearest-neighbour by Tanimoto similarity** — "find
the most similar molecule you have already measured, and copy its number". On
congeneric ChEMBL sets this is genuinely hard to beat, and a model that fails to beat
it has learned nothing that a lookup table does not already contain. It is also the
baseline that is almost always omitted, which is why so many published QSAR models
are, functionally, slow lookup tables.

Every model is scored with **95% bootstrap confidence intervals**. A test set of a
few hundred compounds gives an R² whose interval is routinely ±0.15 wide; quoting a
bare "R² = 0.62" from such a set and ranking models by it is ranking noise.

In [ ]:
featurizer = features.get_featurizer(CFG.get('features.primary'), CFG.get('features.n_bits'))
X = features.featurize_frame(clean, featurizer)
y = clean['pIC50'].to_numpy(float)

tr, va, te = split_idx['train'], split_idx['val'], split_idx['test']
fit_idx = np.concatenate([tr, va])          # final fit uses train + val
print('feature matrix:', X.shape)

In [ ]:
N_BOOT = CFG.get('evaluation.bootstrap.n_resamples', 1000)
NOISE_FLOOR = CFG.get('evaluation.assay_noise.rmse_floor', 0.68)
ZOO = CFG.get('models.zoo')

def benchmark(idx_map, label):
    boots, preds = {}, {}
    a = np.concatenate([idx_map['train'], idx_map['val']]); b = idx_map['test']
    for name in ZOO:
        est = models.get_model(name, seed=SEED, n_jobs=-1).fit(X[a], y[a])
        p = est.predict(X[b])
        boots[name] = evaluate.bootstrap_metrics(y[b], p, N_BOOT, seed=SEED)
        preds[name] = p
    return boots, preds

boots_scaf, preds_scaf = benchmark(split_idx,  'scaffold')
boots_rand, _          = benchmark(random_idx, 'random')

rows = []
for name in ZOO:
    t = evaluate.metrics_table(boots_scaf[name])
    rows.append({'model': models.MODEL_DISPLAY_NAMES.get(name, name),
                 'RMSE': t['RMSE'], 'R2': t['R2'], 'Spearman rho': t['Spearman_rho'],
                 'within +/-1 log (%)': t['pct_within_1.0']})
pd.DataFrame(rows)

In [ ]:
comp_scaf = evaluate.compare_models(boots_scaf, 'RMSE')
comp_rand = evaluate.compare_models(boots_rand, 'RMSE')

plots.plot_model_forest(comp_scaf, FIG, 'RMSE', 'knn_tanimoto', NOISE_FLOOR,
                        models.MODEL_DISPLAY_NAMES, ('png',))
display(Image(str(FIG / 'fig09_model_forest_rmse.png')))

### How much does a random split flatter the model?

The bars below are the same models scored two ways. The Δ annotation is the amount of
performance that a random split invents.

In [ ]:
plots.plot_split_comparison(comp_scaf, comp_rand, FIG, 'RMSE',
                            models.MODEL_DISPLAY_NAMES, ('png',))
display(Image(str(FIG / 'fig10_split_comparison.png')))

delta = comp_scaf.merge(comp_rand, on='model', suffixes=('_scaf','_rand'))
delta['optimism'] = delta['value_rand'] - delta['value_scaf']
print(delta[['model','value_scaf','value_rand','optimism']].round(3).to_string(index=False))

<a id='6'></a>
## 6. Molecular-representation ablation

Does the representation matter more than the algorithm? Six featurizations, one
model, intervals on everything. MACCS (167 hand-designed keys) is included as a
deliberately weak control: if a 2048-bit circular fingerprint cannot beat it, the
extra 1881 dimensions are not buying anything.

In [ ]:
abl_rows = []
for fname in CFG.get('features.ablation'):
    f = features.get_featurizer(fname, CFG.get('features.n_bits'))
    Xa = f.transform(clean['smiles'].tolist())
    est = models.get_model('xgboost', seed=SEED, n_jobs=-1).fit(Xa[fit_idx], y[fit_idx])
    b = evaluate.bootstrap_metrics(y[te], est.predict(Xa[te]), N_BOOT, seed=SEED)
    r  = b.loc[b['metric'] == 'RMSE'].iloc[0]
    r2 = b.loc[b['metric'] == 'R2'].iloc[0]
    abl_rows.append({'featurizer': fname, 'n_features': Xa.shape[1],
                     'value': r['value'], 'ci_low': r['ci_low'], 'ci_high': r['ci_high'],
                     'R2': r2['value']})

ablation = pd.DataFrame(abl_rows)
plots.plot_feature_ablation(ablation, FIG, 'RMSE', ('png',))
display(Image(str(FIG / 'fig19_feature_ablation.png')))
ablation.round(3)

<a id='7'></a>
## 7. Validation: model selection, nested CV, y-scrambling, split stability

Four checks, each closing a different way of fooling yourself.

**Model selection on validation.** The primary model is chosen by validation RMSE.
Never by test score.

**Nested cross-validation.** Tuning hyperparameters inside the same folds you score
on leaks the answer into the tuning and inflates the result. Here the inner search
never sees the outer fold it is evaluated on, and the outer folds are
scaffold-disjoint so the CV estimate matches the split protocol.

**Y-scrambling (OECD QSAR validation principle 4).** Shuffle the training labels and
refit. A model learning real structure–activity signal collapses to R² ≈ 0. One that
is exploiting leakage or memorising noise keeps scoring well. The empirical p-value is
the fraction of permuted runs that match or beat the real model.

**Multi-seed repeats.** A single split's score is a random variable. Its spread across
seeds is frequently larger than the gap between two models being compared — which is
why reporting one number from one split, as the legacy notebook did, is not a result.

In [ ]:
from sklearn.model_selection import GroupKFold

primary = CFG.get('models.primary')
if str(primary).lower() == 'auto':
    # Scaffold-grouped CV over train+val. Never the test set -- and not a single
    # 60-compound validation slice either: that estimator is noisy enough to pick
    # a model that then underperforms, i.e. selection adding variance rather than
    # removing it.
    groups_fit = clean['scaffold'].to_numpy()[fit_idx]
    gkf = GroupKFold(n_splits=min(5, len(np.unique(groups_fit))))
    cv_rmse, cv_sd = {}, {}
    for name in ZOO:
        if name == 'dummy_mean':
            continue
        folds = []
        for f_tr, f_te in gkf.split(X[fit_idx], y[fit_idx], groups=groups_fit):
            est = models.get_model(name, seed=SEED, n_jobs=-1).fit(X[fit_idx][f_tr], y[fit_idx][f_tr])
            folds.append(float(np.sqrt(np.mean((y[fit_idx][f_te] - est.predict(X[fit_idx][f_te]))**2))))
        cv_rmse[name], cv_sd[name] = float(np.mean(folds)), float(np.std(folds, ddof=1))
    primary = min(cv_rmse, key=cv_rmse.get)
    display(pd.DataFrame({'CV RMSE': cv_rmse, 'SD across folds': cv_sd}).sort_values('CV RMSE').round(3))

print(f'primary model selected by scaffold-grouped CV: {primary}')

final = models.get_model(primary, seed=SEED, n_jobs=-1).fit(X[fit_idx], y[fit_idx])
y_pred = final.predict(X[te])
boot_final = evaluate.bootstrap_metrics(y[te], y_pred, N_BOOT, seed=SEED)
mtext = evaluate.metrics_table(boot_final)
pd.DataFrame([mtext]).T.rename(columns={0: 'test (95% CI)'})

In [ ]:
ncv = evaluate.nested_cv(
    models.get_model(primary, seed=SEED, n_jobs=-1), X, y,
    models.get_param_distributions(primary), model_name=primary,
    outer_folds=CFG.get('models.nested_cv.outer_folds', 5),
    inner_folds=CFG.get('models.nested_cv.inner_folds', 3),
    n_iter=CFG.get('models.nested_cv.n_iter', 30),
    seed=SEED, groups=clean['scaffold'].to_numpy())

ncv.summary().set_index('metric').loc[['RMSE','R2','MAE','Spearman_rho']].round(3)

In [ ]:
ys = evaluate.y_scrambling(
    models.get_model(primary, seed=SEED, n_jobs=-1),
    X[fit_idx], y[fit_idx], X[te], y[te],
    n_permutations=CFG.get('evaluation.y_scrambling.n_permutations', 100), seed=SEED)

print(f"real model R2      : {ys['real_r2']:.3f}")
print(f"permuted-label R2  : {ys['null_r2_mean']:.3f} +/- {ys['null_r2_std']:.3f}")
print(f"empirical p-value  : {ys['p_value']:.4f}")

plots.plot_y_scrambling(ys, FIG, ('png',))
display(Image(str(FIG / 'fig11_y_scrambling.png')))

In [ ]:
def _split_fn(d, s):
    return splits.scaffold_split(d, CFG.get('split.test_size'), CFG.get('split.val_size'),
                                 balanced=True, seed=s)

reps = evaluate.repeated_split_eval(
    models.get_model(primary, seed=SEED, n_jobs=-1), clean,
    lambda d: featurizer.transform(d['smiles'].tolist()), _split_fn,
    n_seeds=CFG.get('evaluation.repeats.n_seeds', 10), base_seed=SEED)

plots.plot_seed_stability(reps, FIG, ('png',))
display(Image(str(FIG / 'fig20_seed_stability.png')))
print(reps[['RMSE','R2','MAE','Spearman_rho']].agg(['mean','std']).round(3).to_string())

In [ ]:
lc = evaluate.learning_curve(
    models.get_model(primary, seed=SEED, n_jobs=-1),
    X[fit_idx], y[fit_idx], X[te], y[te],
    fractions=CFG.get('evaluation.learning_curve.fractions'),
    n_repeats=CFG.get('evaluation.learning_curve.n_repeats', 5), seed=SEED)

plots.plot_learning_curve(lc, FIG, NOISE_FLOOR, ('png',))
display(Image(str(FIG / 'fig08_learning_curve.png')))

<a id='8'></a>
## 8. Applicability domain and conformal prediction

Two questions a project team will actually ask, neither of which an R² answers.

**"Can I use this model on my compound?"** The applicability domain is the mean
Tanimoto similarity to the *k* nearest training compounds, with the boundary set from
the training set's own distribution. The useful output is not the in/out flag — it is
the **error-versus-distance curve**, which shows quantitatively how fast predictions
degrade as you leave the training neighbourhood.

**"How wrong is this specific prediction likely to be?"** Split-conformal prediction
turns held-out calibration residuals into an interval with a distribution-free
coverage guarantee. One caveat is stated up front: that guarantee assumes
exchangeability, and a scaffold split *deliberately* violates it. Empirical coverage
is therefore expected to fall short of nominal, and reporting the shortfall is more
useful than asserting a guarantee that does not hold.

In [ ]:
fps_all = features.rdkit_fps(clean['smiles'].tolist())
ad = evaluate.ApplicabilityDomain(k=CFG.get('evaluation.applicability_domain.k_neighbors', 5))
ad.fit([fps_all[i] for i in fit_idx])

test_fps = [fps_all[i] for i in te]
ad_scores = ad.score(test_fps)
ad_profile = ad.error_profile(test_fps, y[te], y_pred,
                              n_bins=CFG.get('evaluation.applicability_domain.n_bins', 5), seed=SEED)

print(f"domain threshold          : {ad.threshold_:.3f}")
print(f"test compounds out-of-domain: {(ad_scores < ad.threshold_).mean():.1%}")
display(ad_profile.round(3))

plots.plot_applicability_domain(ad_profile, ad.train_scores_, ad_scores, ad.threshold_, FIG, ('png',))
display(Image(str(FIG / 'fig12_applicability_domain.png')))

### The headline figure

Equal axes, an explicit ±1 log-unit band, points coloured by applicability-domain distance, and metrics quoted with intervals. Compare with the legacy Yellowbrick plot, whose auto-scaled axes were stretched to pIC50 = 16 by a single corrupt point.

In [ ]:
plots.plot_pred_vs_obs(y[te], y_pred, mtext, FIG, ad_scores,
                       title=f'Predicted vs observed — {models.MODEL_DISPLAY_NAMES.get(primary, primary)}, scaffold-split test set',
                       noise_floor=NOISE_FLOOR, formats=('png',))
display(Image(str(FIG / 'fig06_pred_vs_obs.png')))

In [ ]:
plots.plot_residual_diagnostics(y[te], y_pred, FIG, NOISE_FLOOR, ('png',))
display(Image(str(FIG / 'fig07_residual_diagnostics.png')))

In [ ]:
cov = evaluate.coverage_curve(models.get_model(primary, seed=SEED, n_jobs=-1),
                              X[fit_idx], y[fit_idx], X[te], y[te], seed=SEED)
conf = evaluate.conformal_intervals(models.get_model(primary, seed=SEED, n_jobs=-1),
                                    X[fit_idx], y[fit_idx], X[te],
                                    alpha=CFG.get('conformal.alpha', 0.2), seed=SEED)
emp = float(((y[te] >= conf['lower']) & (y[te] <= conf['upper'])).mean())
print(f"nominal coverage  : {1 - conf['alpha']:.0%}")
print(f"empirical coverage: {emp:.1%}")
print(f"interval half-width: +/-{conf['half_width']:.3f} log units")

plots.plot_conformal_calibration(cov, FIG, ('png',))
display(Image(str(FIG / 'fig13_conformal_calibration.png')))

<a id='9'></a>
## 9. Interpretation: SHAP and activity cliffs

A fingerprint model that cannot be interrogated is a black box no medicinal chemist
will act on. Two things make it actionable.

**SHAP mapped back to chemistry.** A bit index is meaningless on its own, so each
influential bit is traced to a molecule that sets it and the atom environment that
does so — "bit 1174" becomes a highlighted substructure.

**Activity cliffs.** Pairs of near-identical molecules with order-of-magnitude
potency differences are where every similarity-based model fails. Enumerating them
converts "the model has some large errors" into "here are the specific pairs that
cause them", and pre-empts the obvious interview question.

In [ ]:
feat_names = [f'bit_{i}' for i in range(X.shape[1])]
sv, shap_meta = interpret.compute_shap(final, X[fit_idx], X[te],
                                       background_size=CFG.get('interpretation.shap.background_size', 100), seed=SEED)
print('attribution method:', shap_meta['method'])
if shap_meta['surrogate_fidelity_r2'] is not None:
    print(f"surrogate fidelity : R2 = {shap_meta['surrogate_fidelity_r2']:.3f} vs the primary model")
imp = interpret.shap_importance(sv, feat_names)

plots.plot_shap_bar(imp, FIG, 20, ('png',))
display(Image(str(FIG / 'fig15_shap_importance.png')))
imp.head(10).round(4)

In [ ]:
recs = interpret.top_bit_substructures(imp, clean.iloc[fit_idx]['smiles'].tolist(), featurizer, top_n=6)
if recs:
    plots.plot_bit_substructures(recs, FIG)
    display(Image(str(FIG / 'fig16_shap_substructures.png')))
else:
    print('No ECFP bits available for substructure mapping with this featurizer.')

In [ ]:
cl = CFG.get('interpretation.activity_cliffs', {})
pairs, sim_flat, delta_flat = interpret.find_activity_cliffs(
    clean, cl.get('similarity_threshold', 0.7), cl.get('potency_delta', 2.0))

plots.plot_cliff_landscape(sim_flat, delta_flat, FIG,
                           cl.get('similarity_threshold', 0.7), cl.get('potency_delta', 2.0), ('png',))
display(Image(str(FIG / 'fig18_cliff_landscape.png')))

if len(pairs):
    resid_map = dict(zip(clean.iloc[te]['chembl_id'], y[te] - y_pred))
    cliff_err = interpret.cliff_error_analysis(pairs, set(clean.iloc[te]['chembl_id']), resid_map)
    display(cliff_err.round(3))
    plots.plot_activity_cliffs(pairs, FIG, cl.get('max_pairs_plotted', 6))
    display(Image(str(FIG / 'fig17_activity_cliffs.png')))

<a id='10'></a>
## 10. Results, limitations, and what I would do next

### Results

In [ ]:
print('=' * 74)
print(f'DRD2 QSAR — {models.MODEL_DISPLAY_NAMES.get(primary, primary)} on a scaffold-split test set')
print('=' * 74)
print(f"compounds (curated)   : {len(clean):,}")
print(f"test set              : {len(te)}")
print(f"scaffolds             : {clean['scaffold'].nunique()}")
print()
for k in ['R2','RMSE','MAE','Spearman_rho','pct_within_1.0']:
    print(f'{k:<22}: {mtext[k]}')
print()
print(f"assay noise floor     : RMSE ~ {NOISE_FLOOR:.2f}  ({CFG.get('evaluation.assay_noise.source')})")
print(f"y-scrambling p-value  : {ys['p_value']:.4f}")
print(f"out-of-domain (test)  : {(ad_scores < ad.threshold_).mean():.1%}")
print(f"conformal {1-conf['alpha']:.0%} interval : +/-{conf['half_width']:.2f} log units "
      f"(empirical coverage {emp:.1%})")
print('=' * 74)

### Limitations — stated, not buried

1. **The temporal result, not the scaffold result, is the deployment number.** Trained
   on pre-2015 literature and tested on 2015-onward chemistry, performance degrades
   substantially. Quote that number when describing expected prospective behaviour.
2. **Censored measurements are discarded.** Rows reported as `IC50 > x` are not
   regression targets, but they are overwhelmingly the inactives — so the model is
   weaker at recognising inactivity than its scaffold-split metrics suggest. This is
   the largest piece of usable signal currently left on the table.
3. **Potency truncation.** If the legacy `IC50 < 1000 nM` window is re-enabled
   (`curation.apply_activity_cutoff: true`), the model never sees an inactive compound,
   pIC50 variance collapses, and R² becomes structurally near zero regardless of model
   quality. The default configuration keeps the full dynamic range.
4. **Mechanistic heterogeneity.** ChEMBL's mechanism annotations cover only
   well-studied compounds, so most of the set is unlabelled. Agonists and antagonists
   may still be pooled. `action_type_policy` mitigates this; it does not solve it.
5. **Congeneric-series bias.** A large fraction of compounds sit on a handful of
   scaffolds. The scaffold split addresses the evaluation, not the underlying
   coverage: whole regions of DRD2-relevant chemistry are simply absent.
6. **Heterogeneous assay provenance.** Pooled multi-laboratory IC50 data carries a
   reproducibility floor of roughly 0.7 log units. No model can beat it, and one that
   appears to has almost certainly leaked.
7. **Exchangeability.** The conformal guarantee is marginal and assumes
   exchangeability, which a scaffold split violates by design.
8. **2D only.** No conformational, protein-structural or free-energy information is
   used. DRD2 has experimental structures (e.g. PDB 6CM4) that a structure-based
   arm could exploit.

### What I would do next

1. **Use the censored data.** The discarded `IC50 > x` rows are the largest available
   gain: a classification arm, or a censored-regression model that treats them as
   lower bounds, is what turns this into a screening model rather than a
   series-ranking one.
2. **Multi-task learning across DRD1–DRD5** to exploit the aminergic GPCR family, with
   DRD2/DRD3 selectivity as a second endpoint. Selectivity is the real
   medicinal-chemistry problem here.
3. **A graph neural network (Chemprop D-MPNN)** as a learned-representation arm of the
   ablation, compared on the same scaffold split with the same intervals.
4. **Docking cross-check against DRD2 (6CM4)** for the highest-error compounds, tying
   the ligand-based model to structure.
5. **Uncertainty-guided active learning** — use the conformal intervals to choose which
   compounds to measure next, which is where this model earns its keep in a real
   project.

---

### Reproducing this work

    git clone https://github.com/Babakmamnoon/QSAR-Modelling-of-Human-Dopamine-Receptor-D2-DRD2-Ligand-Potency
    cd QSAR-Modelling-of-Human-Dopamine-Receptor-D2-DRD2-Ligand-Potency
    conda env create -f environment.yml && conda activate drd2-qsar
    python -m drd2_qsar.run --config configs/drd2.yaml
    pytest tests -q

Predicting on new compounds:

    python -m drd2_qsar.predict --model models/drd2_qsar_model.joblib \
        --input my_compounds.smi --output predictions.csv

Every prediction comes back with an applicability-domain flag, a conformal interval,
and its nearest training neighbour — because a bare number is not a deliverable.